# 2장 1강: 역전파와 기울기 문제 — 실습문제


## 실습 목표

- 순전파, 손실 계산, 역전파, 파라미터 수정의 역할을 구분합니다.
- 주어진 미분 식에 숫자를 대입하여 가중치·편향의 기울기를 구합니다.
- 기울기를 이용해 파라미터를 한 번 수정하고, 같은 샘플의 손실을 다시 확인합니다.
- 기울기가 여러 계산 단계를 거치며 너무 작아지거나 커지는 상황을 설명합니다.

## 실습 환경 / 데이터

- Python, pandas, NumPy를 사용합니다. pandas는 CSV를 읽고 NumPy는 숫자 계산을 돕습니다.
- 이전 강의와 같은 **`concrete_data.csv`**를 사용합니다. CSV와 노트북을 같은 폴더에 놓고, 해당 폴더를 VS Code에서 열어 실행하세요.
- pandas와 NumPy가 설치된 Python 커널을 선택하고, 실습 준비부터 위에서 아래로 실행하세요.
- 원본은 1030개 샘플과 9개 열입니다. 입력 후보는 8개 특성이고 실제 정답은 `concrete_compressive_strength`입니다.
- 손계산을 위해 이번 모델은 **시멘트량 한 특성만** 사용합니다. 나머지 특성이 불필요하다는 결론이 아닙니다.
- 입력은 `시멘트량 / 100`으로 정합니다. 계산에 쓰는 숫자를 줄이는 **고정 배율 변환**이며 데이터에서 평균·표준편차를 학습하는 과정은 아닙니다. 실제 압축강도 정답은 나누지 않고 원래 값을 씁니다.
- 실습은 첫 샘플, 과제는 세 번째 샘플을 사용합니다. 전체 데이터의 예측 성능을 평가하는 과제가 아닙니다.

## 이번 강의의 범위

제공된 **2장 1강 목차: 역전파와 파라미터 학습, 기울기 소실과 폭주**에 맞춘 자료입니다. 강의 본문은 별도로 제공되지 않았으므로 모델 크기·파라미터·손실 예제는 개념 설명용으로 정했습니다.

학습 흐름을 이해하는 데 필요한 손실 식, 미분 식, 한 번의 파라미터 수정 식은 모두 제공합니다. 공식을 새로 유도하거나 복잡한 미분을 풀 필요는 없습니다. 학습률 탐색, 여러 최적화 알고리즘, 정규화 기법, 자동미분 프레임워크, 성능 튜닝은 요구하지 않습니다.

기울기 소실·폭주는 **단순화한 전달 계수 예제**로 확인합니다. 실제 콘크리트 모델에서 해당 문제가 발생했다고 주장하는 실험은 아닙니다.

## 실습 구성

| 구분 | 문제 | 할 일 | 연결되는 강의 내용 |
|---|---|---|---|
| 실습 1 | 1-1 | 순전파·손실 계산 후 네 파라미터의 기울기 구하기 | 역전파 |
| 실습 1 | 1-2 | 파라미터 한 번 수정하고 같은 샘플의 손실 비교 | 파라미터 학습 |
| 실습 2 | 2-1 | 기울기 전달 계수의 반복 곱 비교 | 기울기 소실·폭주 |
| 기초 과제 | 3-1 | 세 번째 샘플에 같은 역전파·수정 적용 | 실습 1·2의 기초 복습 |

**실습 2개 + 기초 과제 1개**입니다. 실습 1은 기울기 계산과 실제 수정을 구분하기 위해 두 세부 문제로 나눴습니다.

## 진행 안내

1. 코드뿐 아니라 ‘수식 및 계산 과정’과 해석 질문에도 답하세요.
2. **순전파 → 손실 → 역전파로 기울기 계산 → 파라미터 수정 → 다시 순전파** 순서로 진행하세요.
3. 기울기는 모두 수정 전의 같은 파라미터로 계산합니다. 일부 값을 먼저 바꾼 뒤 나머지 기울기를 구하지 않습니다.
4. 손계산은 기호에 실제 숫자를 넣고 곱셈·덧셈 순서로 적으면 됩니다.
5. 출력할 때만 반올림하고, 다음 계산에는 반올림 전 값을 사용하세요.

## 시작 전에 읽을 용어

| 용어 | 쉽게 풀어쓴 뜻 | 이번 실습 |
|---|---|---|
| 순전파 | 입력에서 최종 출력까지 계산하는 과정 | x → z1 → h1 → y_hat |
| 실제 정답 y | 데이터에 기록된 측정값 | 첫 샘플 79.99 |
| 예측 출력 y_hat | 모델 식을 계산해 얻은 값 | 수정 전 첫 출력 76 |
| 오차 error | 예측 출력에서 실제 정답을 뺀 값 | y_hat - y |
| 손실 loss | 예측이 정답과 얼마나 다른지 정한 규칙으로 계산한 수 | 오차 제곱의 절반 |
| 기울기 gradient | 어떤 값을 조금 바꿀 때 손실이 어느 방향으로 얼마나 변하는지 나타내는 값 | 가중치별·편향별 기울기 |
| 미분 | 작은 변화에 대한 변화율을 구하는 계산 | 이번에는 완성된 식에 대입 |
| 역전파 | 출력 쪽에서 입력 쪽으로 계산을 거슬러 기울기를 구하는 방법 | 손실 → 출력층 → 은닉층 |
| 파라미터 | 모델에서 조정하는 가중치·편향 | w1, b1, w2, b2 |
| 학습률 lr | 기울기를 얼마나 반영해 값을 바꿀지 정하는 수 | 고정값 0.0001 |
| 기울기 소실 | 역으로 전달되며 기울기가 매우 작아지는 현상 | 작은 전달 계수의 반복 곱 |
| 기울기 폭주 | 역으로 전달되며 기울기가 매우 커지는 현상 | 큰 전달 계수의 반복 곱 |

기울기가 양수라고 파라미터 자체가 양수라는 뜻은 아닙니다. **파라미터의 값과 그 파라미터에 대한 손실의 기울기는 서로 다른 수**입니다.


### 앞 강의와 연결하기 — 이번에는 무엇을 더 하나요?

1장 2강에서는 주어진 가중치와 편향으로 출력을 계산했습니다. 이번에는 출력이 실제 정답과 얼마나 다른지 **손실**로 나타내고, 그 손실을 줄이기 위해 가중치와 편향을 어느 방향으로 조금 바꿀지 알아봅니다.

| 단계 | 사용하는 정보 | 얻는 결과 | 파라미터가 바뀌나요? |
|---|---|---|---|
| 순전파 | 입력, 현재 가중치·편향 | 모델의 출력 | 아직 바뀌지 않음 |
| 손실 계산 | 모델 출력과 실제 정답 | 차이를 나타내는 손실 한 값 | 아직 바뀌지 않음 |
| 역전파 | 순전파 중간값, 현재 파라미터, 손실의 미분 식 | 각 파라미터에 대한 기울기 | 아직 바뀌지 않음 |
| 파라미터 수정 | 기존 값, 기울기, 학습률 | 새 가중치·편향 | 이 단계에서 바뀜 |
| 다시 순전파·손실 계산 | 같은 입력·정답과 새 파라미터 | 수정 후 출력과 손실 | 수정 효과를 확인 |

‘역전파’는 실제 정답을 거꾸로 넣어서 정답 가중치를 한 번에 찾아내는 계산이 아닙니다. **현재 상태에서 작은 변화가 손실에 미치는 영향을 뒤쪽부터 구하는 과정**입니다.

### 기울기를 처음 배운다면

어떤 파라미터 하나를 아주 조금 늘리고 다른 조건은 그대로 두었을 때, 손실이 늘어나는지 줄어드는지 생각해 보세요. 기울기는 그 순간의 변화 방향과 변화율을 나타냅니다.

- 기울기가 **양수**: 그 파라미터를 조금 늘리면 손실이 증가하는 방향입니다.
- 기울기가 **음수**: 그 파라미터를 조금 늘리면 손실이 감소하는 방향입니다.
- 기울기가 **0**: 그 지점에서 아주 작은 변화에 대한 1차 변화율이 0입니다. 이것만으로 가장 좋은 모델이라고 판단하지는 않습니다.

이 설명은 **현재 지점 주변의 작은 변화**에 관한 것입니다. 기울기가 음수라는 이유로 파라미터를 아무리 크게 늘려도 계속 좋아진다는 뜻은 아닙니다.

### 수식과 코드의 표기 읽기

| 표시 | 쉬운 뜻 | 주의할 점 |
|---|---|---|
| $hat y$, `y_hat` | 모델이 계산한 출력 | 실제 정답 y와 다른 값 |
| L, `loss` | 이번에 정한 손실 | 음수일 수 있는 오차 error와 구분 |
| $partial L/partial w1$ | w1을 조금 바꿀 때 손실 L이 어떻게 변하는지 | ‘w1에 대한 손실의 편미분’이라고 읽음. L을 w1로 단순히 나누라는 뜻이 아님 |
| `grad_w1` | 위 기울기에 붙인 코드 이름 | grad는 gradient의 줄임말이며 w1의 새 값이 아님 |
| $a^2$, `a ** 2` | a × a | a × 2와 다름 |
| $c^n$, `c ** n` | c를 n번 반복해서 곱함 | c × n과 다름 |
| 절댓값 | 부호를 제외한 크기 | -3.99와 +3.99의 절댓값은 둘 다 3.99 |
| `lr` | learning rate, 학습률 | 이번에는 고정된 0.0001을 사용 |

미분 공식을 외워서 유도하는 과제가 아닙니다. 제공된 식에 숫자를 대입하면서 ‘이 수는 출력인가, 기울기인가, 새 파라미터인가’를 구분하세요.


## 실습 준비 — 제공 코드를 그대로 실행하세요

원본의 일부 열 이름에는 끝 공백이 있습니다. 아래 코드는 열 이름의 앞뒤 공백만 제거합니다. 입력값을 100으로 나누는 작업은 각 문제에서 별도로 진행합니다.


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("concrete_data.csv")
df.columns = df.columns.str.strip()
print("전체 데이터 크기:", df.shape)
print(df[["cement", "concrete_compressive_strength"]].head(3).to_string(index=False))


---

## 실습 1. 역전파로 기울기를 구하고 파라미터 수정하기


### 문제 1-1. 첫 샘플의 순전파·손실·기울기 계산

#### 문제 설명

첫 샘플의 시멘트량 540을 100으로 나눈 **x=5.4**, 실제 압축강도 **y=79.99**를 사용합니다. 한 특성으로 실제 현상을 충분히 설명하는 모델을 만드는 것이 아니라, 역전파의 계산 순서를 배우는 문제입니다.

#### 작은 신경망과 주어진 식

구조는 **입력 1개 → ReLU 은닉 뉴런 1개 → 출력 뉴런 1개**입니다. 출력층은 계산값을 그대로 사용합니다.

| 기호 | 뜻 | 초기값 |
|---|---|---:|
| w1 | 입력에서 은닉 뉴런으로 연결되는 가중치 | 0.5 |
| b1 | 은닉 뉴런의 편향 | 0.1 |
| w2 | 은닉값에서 출력 뉴런으로 연결되는 가중치 | 20.0 |
| b2 | 출력 뉴런의 편향 | 20.0 |

순전파와 손실은 다음 식을 사용합니다.

$$z1=w1\times x+b1,\quad h1=\operatorname{ReLU}(z1),\quad \hat y=w2\times h1+b2$$
$$error=\hat y-y,\qquad L=\frac12(\hat y-y)^2$$

여기서 L은 **한 샘플의 오차 제곱에 1/2을 곱한 손실**입니다. 여러 샘플의 오차를 평균한 값은 아닙니다. 앞에 1/2을 둔 이유는 아래 미분 식을 간단하게 만들기 위해서입니다. 오차는 음수일 수 있지만 제곱한 손실은 음수가 되지 않습니다.

#### 제공된 역전파 식 — 증명 없이 대입하세요

`grad_이름`은 그 값에 대한 손실의 기울기입니다. 예를 들어 `grad_w2`는 w2를 조금 바꿀 때 손실이 어떻게 변하는지 나타냅니다. 수식의 $\partial L/\partial w2$를 코드에서는 `grad_w2`라고 적습니다.

| 계산 순서 | 코드로 읽는 식 | 설명 |
|---|---|---|
| 1 | grad_y_hat = y_hat - y | 손실을 출력에 대해 미분한 값 |
| 2 | grad_w2 = grad_y_hat * h1 | 출력 가중치의 기울기 |
| 3 | grad_b2 = grad_y_hat | 출력 편향의 기울기 |
| 4 | grad_h1 = grad_y_hat * w2 | 은닉값 쪽으로 전달하는 기울기 |
| 5 | grad_z1 = grad_h1 * relu_slope | ReLU의 미분값을 반영 |
| 6 | grad_w1 = grad_z1 * x | 은닉 가중치의 기울기 |
| 7 | grad_b1 = grad_z1 | 은닉 편향의 기울기 |

ReLU는 z1>0에서 입력을 그대로 통과시키므로 `relu_slope=1`입니다. z1<0에서는 출력이 계속 0이므로 미분값은 0입니다. z1=0에서는 미분이 정의되지 않으며 **이번 계산에는 그 경우가 나오지 않습니다.** 이 실습의 z1은 양수라 제공 코드에서 `relu_slope=1.0`을 사용합니다.

뒤쪽에서 온 기울기에 해당 단계의 변화율을 곱해 앞쪽으로 전달하는 방식입니다. 연결된 함수의 미분을 곱한다는 원리를 사용하지만, 지금은 표의 순서대로 값을 대입하면 됩니다.

#### 읽기 도움말 — 왜 손실이 필요한가요?

모델 출력이 정답보다 작으면 error는 음수이고, 크면 양수입니다. 이 오차를 제곱하면 두 방향 모두 양수가 되어 ‘얼마나 떨어졌는가’를 비교할 수 있습니다. 예를 들어 오차 -2와 +2는 모두 같은 크기의 틀림으로 보고, 이번 손실에서는 각각 0.5 × 4 = 2가 됩니다.

손실 한 값이 있다고 해서 네 파라미터를 모두 같은 양만큼 바꾸지는 않습니다. 각 파라미터가 출력에 연결되는 방식이 다르므로, 그 영향을 기울기로 따로 구합니다. 역전파 표의 `grad_` 값들을 기존 가중치와 편향에 덮어쓰지 마세요.

#### 요구사항

1. 제공 코드의 x, y, 네 파라미터를 확인하세요.
2. z1, h1, y_hat을 계산하고 출력하세요.
3. `error = y_hat - y`, `loss = 0.5 * error ** 2`를 계산하고 출력하세요.
4. 표의 순서대로 출력층에서 은닉층 방향으로 기울기를 계산하세요.
5. grad_y_hat, grad_h1, grad_z1과 네 파라미터의 기울기를 출력하세요.
6. 수식에 숫자를 넣은 과정을 답변란에 적으세요. 이 문제에서는 파라미터를 바꾸지 않습니다.

#### 작성 도움말

`** 2`는 제곱입니다. `np.maximum(0, z1)`은 ReLU입니다. 이번에는 각 값이 숫자 하나이므로 행렬곱 `@`가 아니라 일반 곱셈 `*`를 사용합니다. `round(값, 6)`으로 소수점 여섯째 자리까지 출력할 수 있습니다.

#### 해석 질문

**Q1.** z1, h1, y_hat, error, loss는 각각 얼마인가요? 음수 error는 무엇을 뜻하나요?

**Q2.** 출력층의 grad_w2와 grad_b2는 각각 얼마인가요? 왜 h1을 곱하는 것은 가중치 쪽인가요?

**Q3.** grad_h1, grad_z1, grad_w1, grad_b1은 각각 얼마인가요? 왜 수정 전 w2를 사용하나요?

**Q4.** 모든 파라미터 기울기가 음수라는 것은 파라미터 값도 음수라는 뜻인가요? 기울기와 실제 정답의 역할은 어떻게 다른가요?

**Q5.** 역전파로 기울기를 구한 시점에 파라미터 수정도 끝난 것인가요?

#### 제출 결과

- 순전파·오차·손실 출력
- 중간 기울기와 네 파라미터 기울기 출력
- 대입 계산 과정 및 Q1~Q5 답변


In [ ]:
# 데이터와 초기값을 준비하는 제공 코드입니다.
x = df["cement"].to_numpy()[0] / 100.0
y = df["concrete_compressive_strength"].to_numpy()[0]

w1 = 0.5
b1 = 0.1
w2 = 20.0
b2 = 20.0

# 아래부터 작성하세요.
# 순전파 → error·loss → 제공된 역전파 식 순서로 진행합니다.
# z1이 양수인지 확인한 뒤 relu_slope = 1.0을 사용하세요.
# 파라미터는 이 문제에서 수정하지 않습니다.


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. z1, h1, y_hat, error, loss는 각각 얼마인가요? 음수 error는 무엇을 뜻하나요?**

- 답변: 

**Q2. 출력층의 grad_w2와 grad_b2는 각각 얼마인가요? 왜 h1을 곱하는 것은 가중치 쪽인가요?**

- 답변: 

**Q3. grad_h1, grad_z1, grad_w1, grad_b1은 각각 얼마인가요? 왜 수정 전 w2를 사용하나요?**

- 답변: 

**Q4. 모든 파라미터 기울기가 음수라는 것은 파라미터 값도 음수라는 뜻인가요? 기울기와 실제 정답의 역할은 어떻게 다른가요?**

- 답변: 

**Q5. 역전파로 기울기를 구한 시점에 파라미터 수정도 끝난 것인가요?**

- 답변:


### 문제 1-2. 기울기로 네 파라미터를 한 번 수정하기

#### 문제 설명

문제 1-1에서 구한 기울기를 사용합니다. **문제 1-1을 먼저 실행**한 뒤 진행하세요. 같은 첫 샘플에서 한 번 수정한 결과만 확인합니다.

#### 수식을 말로 읽기

$$\text{새 파라미터}=\text{기존 파라미터}-lr\times\text{해당 파라미터의 기울기}$$

학습률 lr은 기울기를 반영하는 크기이며 이번에는 **0.0001로 고정**합니다. 학습률을 찾아 바꾸는 문제는 아닙니다. 기울기 반대 방향으로 움직이면 충분히 작은 변화에서 손실을 줄이는 방향으로 이동할 수 있습니다. 큰 폭으로 바꾸면 손실이 오를 수도 있으므로 수정 후에는 다시 계산합니다.

- 기울기가 양수이면 양수를 빼므로 파라미터가 작아집니다.
- 기울기가 음수이면 음수를 빼므로 파라미터가 커집니다.
- 기울기가 0이면 이 식에서는 그 파라미터가 바뀌지 않습니다.

#### 읽기 도움말 — 기울기와 실제 변화량은 다릅니다

먼저 ‘학습률 × 기울기’를 계산하고, 그 수를 기존 값에서 뺍니다. 따라서 **새 값에서 기존 값을 뺀 실제 변화량은 −학습률 × 기울기**입니다. 기울기를 그대로 새 파라미터로 사용하지 않습니다.

‘네 값을 동시에 수정한다’는 말은 컴퓨터가 네 줄을 같은 순간에 실행해야 한다는 뜻이 아닙니다. 네 기울기를 모두 같은 수정 전 상태에서 구해 놓고 각각의 새 값을 만든다는 뜻입니다. 수정 후 손실도 같은 입력·같은 실제 정답으로 다시 계산해야 전후를 비교할 수 있습니다.

#### 요구사항

1. `lr = 0.0001`을 준비하세요.
2. 기존 값과 문제 1-1의 기울기로 `w1_new`, `b1_new`, `w2_new`, `b2_new`를 계산하세요.
3. 새 파라미터로 `z1_new`, `h1_new`, `y_hat_new`를 다시 계산하세요.
4. 수정 후 손실 `loss_new = 0.5 * (y_hat_new - y) ** 2`를 계산하세요.
5. 네 파라미터의 수정 전후 값, 출력 전후 값, 손실 전후 값을 출력하세요.
6. `loss_new < loss`를 출력해 감소 여부를 확인하세요.

#### 작성 도움말

`w1_new = w1 - lr * grad_w1`과 같은 형식을 모든 파라미터에 적용합니다. 기존 w1을 직접 덮어쓰지 않고 `_new`라는 이름을 사용하면 수정 전후를 비교하기 쉽습니다. 새 출력은 **네 값 모두 새 파라미터로** 계산하세요. 이전 h1을 재사용하면 안 됩니다.

#### 해석 질문

**Q1.** 네 파라미터의 새 값은 각각 얼마인가요? 음수 기울기를 뺐는데 왜 값이 커지나요?

**Q2.** 수정 전후 출력과 손실은 각각 얼마인가요? 수정 후 출력이 실제 정답보다 커졌는데도 손실이 줄었나요?

**Q3.** 새 출력 계산에서 은닉값 h1도 다시 구해야 하는 이유는 무엇인가요?

**Q4.** 첫 샘플의 손실이 한 번 줄었다는 결과로 전체 데이터에서 좋은 모델이라고 결론 내릴 수 있나요?

#### 제출 결과

- 파라미터·출력·손실의 수정 전후 값
- 손실 감소 여부
- 파라미터 수정 계산 과정 및 Q1~Q4 답변


In [ ]:
# 문제 1-1을 먼저 실행하세요.
lr = 0.0001

# 아래부터 작성하세요.
# 네 파라미터는 기존 값 - lr * 기존 기울기로 동시에 수정합니다.
# 수정 후 순전파와 손실을 다시 계산하고 전후를 출력하세요.


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. 네 파라미터의 새 값은 각각 얼마인가요? 음수 기울기를 뺐는데 왜 값이 커지나요?**

- 답변: 

**Q2. 수정 전후 출력과 손실은 각각 얼마인가요? 수정 후 출력이 실제 정답보다 커졌는데도 손실이 줄었나요?**

- 답변: 

**Q3. 새 출력 계산에서 은닉값 h1도 다시 구해야 하는 이유는 무엇인가요?**

- 답변: 

**Q4. 첫 샘플의 손실이 한 번 줄었다는 결과로 전체 데이터에서 좋은 모델이라고 결론 내릴 수 있나요?**

- 답변:


---

## 실습 2. 기울기 소실과 폭주를 숫자로 확인하기


### 문제 2-1. 역으로 전달되는 기울기의 반복 곱 비교

#### 문제 설명

실습 1에서는 grad_h1=grad_y_hat×w2, grad_z1=grad_h1×relu_slope처럼 뒤쪽의 기울기에 해당 단계의 값을 곱했습니다. 여러 단계가 연결되면 이런 곱셈이 반복됩니다.

이번에는 전달 과정만 보기 위해 **출력 쪽에서 출발하는 기울기를 1**로 고정합니다. 각 단계의 전달 계수가 모두 같다고 가정합니다.

| 경우 | 매 단계 곱하는 계수 | 관찰 목적 |
|---|---:|---|
| 작은 계수 | 0.1 | 반복하며 작아지는 경우 |
| 기준 계수 | 1.0 | 크기가 유지되는 경우 |
| 큰 계수 | 2.0 | 반복하며 커지는 경우 |

이 계수는 가중치와 활성화 함수의 미분 등이 결합된 **단순화한 한 경로의 전달 계수**입니다. 학습률이 아닙니다. 실제 신경망의 모든 단계가 항상 같은 계수라는 뜻도 아닙니다. 이 문제는 콘크리트 데이터를 학습시킨 결과가 아니라 원리 확인용 계산입니다.

#### 수식을 말로 읽기

$$g_{\text{앞쪽}}=g_{\text{출력쪽}}\times c^n$$

- g는 기울기, c는 매번 곱하는 전달 계수입니다.
- n은 거꾸로 지나가는 계산 단계의 수입니다.
- $c^n$은 c를 n번 곱한다는 뜻입니다. 코드에서는 `c ** n`입니다.
- 예: 계수 0.1을 세 번 거치면 1×0.1×0.1×0.1=0.001입니다.

#### 읽기 도움말 — 소실과 폭주에서 변하는 것은 무엇인가요?

여기서 작아지거나 커지는 것은 뒤에서 앞으로 전달되는 **기울기의 크기**입니다. 원본 입력 데이터가 사라지거나 뉴런의 개수가 늘어나는 현상이 아닙니다. ‘앞쪽’은 입력에 가까운 쪽, ‘뒤쪽’은 출력·손실에 가까운 쪽을 뜻합니다.

계수 c는 역전파에서 기울기가 한 단계를 통과할 때 곱해지는 수이고, 학습률 lr은 구한 기울기로 파라미터를 바꿀 때 사용하는 수입니다. 서로 다른 단계에서 다른 역할을 합니다. 이번 단순 예제는 이 전달 과정을 따로 떼어 비교합니다.

#### 요구사항

1. 단계 수 `steps = np.array([1, 3, 5, 10])`과 출발 기울기 `g_start = 1.0`을 준비하세요.
2. `g_small = g_start * (0.1 ** steps)`를 계산하세요.
3. 같은 방법으로 계수 1.0의 `g_same`, 계수 2.0의 `g_large`를 계산하세요.
4. 단계 수와 세 결과 배열을 출력하세요.
5. 계수 0.1과 2.0을 세 번 곱하는 손계산을 각각 적으세요.

#### 작성 도움말

숫자에 `** steps`를 적용하면 각 단계 수에 해당하는 거듭제곱을 계산합니다. 출력 `1e-10`은 $1\times10^{-10}$, 즉 0.0000000001입니다. 작은 수를 0으로 반올림하면 실제 크기를 읽기 어려우므로 이 문제는 **반올림 없이 출력**하세요.

#### 해석 질문

**Q1.** 세 단계를 거친 작은 계수·기준 계수·큰 계수의 결과는 각각 얼마인가요?

**Q2.** 열 단계를 거친 결과는 각각 얼마이며, 어느 경우가 소실·폭주를 설명하나요?

**Q3.** 앞쪽 파라미터에 전달되는 기울기가 너무 작거나 너무 크면 파라미터 수정에 어떤 문제가 생길 수 있나요?

**Q4.** 기울기가 작으면 언제나 소실, 크면 언제나 폭주라고 판단할 수 있나요? 이 예제에서 무엇을 고정해 비교했나요?

**Q5.** ReLU가 양수에서 미분값 1을 갖는다는 사실만으로 소실·폭주가 모두 해결된다고 할 수 있나요?

#### 제출 결과

- 단계 수와 세 기울기 배열
- 세 번 곱하는 손계산 및 Q1~Q5 답변


In [ ]:
steps = np.array([1, 3, 5, 10])
g_start = 1.0

# 아래부터 작성하세요.
# g_start * (계수 ** steps)로 세 경우를 계산하세요.
# 아주 작은 값도 보이도록 반올림 없이 출력하세요.


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. 세 단계를 거친 작은 계수·기준 계수·큰 계수의 결과는 각각 얼마인가요?**

- 답변: 

**Q2. 열 단계를 거친 결과는 각각 얼마이며, 어느 경우가 소실·폭주를 설명하나요?**

- 답변: 

**Q3. 앞쪽 파라미터에 전달되는 기울기가 너무 작거나 너무 크면 파라미터 수정에 어떤 문제가 생길 수 있나요?**

- 답변: 

**Q4. 기울기가 작으면 언제나 소실, 크면 언제나 폭주라고 판단할 수 있나요? 이 예제에서 무엇을 고정해 비교했나요?**

- 답변: 

**Q5. ReLU가 양수에서 미분값 1을 갖는다는 사실만으로 소실·폭주가 모두 해결된다고 할 수 있나요?**

- 답변:


---

## 과제. 같은 역전파와 파라미터 수정 복습


### 문제 3-1. 세 번째 샘플로 한 번의 학습 계산 수행

#### 문제 설명

세 번째 샘플의 시멘트량은 **332.5**, 실제 압축강도는 **40.27**입니다. 계산용 입력은 332.5/100=**3.325**입니다.

이번에도 `w1=0.5, b1=0.1, w2=20, b2=20`에서 출발합니다. **실습에서 수정한 새 파라미터를 이어서 사용하지 않습니다.** 같은 초기 조건에서 다른 샘플에 동일한 방법을 적용하는 복습입니다.

#### 제공된 계산 순서

1. `z1=w1*x+b1`, `h1=ReLU(z1)`, `y_hat=w2*h1+b2`
2. `error=y_hat-y`, `loss=0.5*error**2`
3. `grad_y_hat=error`
4. `grad_w2=grad_y_hat*h1`, `grad_b2=grad_y_hat`
5. `grad_h1=grad_y_hat*w2`, `grad_z1=grad_h1*relu_slope`
6. `grad_w1=grad_z1*x`, `grad_b1=grad_z1`
7. 각 파라미터의 새 값 = 기존 값 - 0.0001 × 해당 기울기
8. 새 파라미터로 순전파와 손실 다시 계산

이번에도 z1이 양수여서 `relu_slope=1.0`을 사용합니다. 모든 미분 식은 실습 1과 같습니다.

#### 읽기 도움말 — 과제의 시작점

입력과 실제 정답은 세 번째 샘플로 바꾸되, 파라미터는 처음 주어진 값으로 다시 시작합니다. 실습에서 얻은 `_new` 값을 가져오지 마세요. 순전파 → 손실 → 역전파 → 한 번 수정 → 다시 계산이라는 순서를 그대로 복습하면 됩니다.

마지막의 반복 곱 계산은 기울기 소실·폭주의 원리 복습입니다. 이 계산의 0.1과 2를 콘크리트 모델의 가중치나 학습률로 넣는 것은 아닙니다.

#### 요구사항

1. 제공 코드로 세 번째 입력·정답과 초기 파라미터를 준비하세요.
2. 수정 전 순전파, 오차, 손실을 구해 출력하세요.
3. 출력층에서 은닉층 방향으로 역전파하고 네 파라미터의 기울기를 출력하세요.
4. 학습률 0.0001로 네 파라미터를 한 번 수정하고 새 값을 출력하세요.
5. 수정 후 출력과 손실을 다시 구하고, 전후 손실과 감소 여부를 출력하세요.
6. 기초 복습으로 출발 기울기 1에 계수 0.1과 2를 각각 **세 번** 곱한 결과를 출력하세요. 이 계산은 콘크리트 모델과 별도의 원리 복습입니다.
7. 손계산 과정과 Q1~Q5 답변을 작성하세요.

#### 작성 도움말

`[2]`가 세 번째 샘플입니다. 변수 이름에 `task_`를 붙여 실습의 값과 구분하세요. 실습 1-1·1-2를 참고해 같은 식을 사용하면 됩니다. 기울기는 모두 수정 전 파라미터에서 계산하고, 반올림은 출력할 때만 합니다.

#### 해석 질문

**Q1.** 수정 전 z1, h1, 출력, 오차, 손실은 각각 얼마인가요?

**Q2.** 네 파라미터의 기울기는 각각 얼마인가요? 실습의 첫 샘플과 부호가 다른 이유는 무엇인가요?

**Q3.** 네 파라미터를 한 번 수정한 값과 수정 후 출력·손실은 얼마인가요? 파라미터가 증가했나요, 감소했나요?

**Q4.** 기울기 1에 계수 0.1과 2를 세 번 곱한 결과는 각각 얼마이며, 어떤 현상을 설명하나요?

**Q5.** 이번 과제가 첫 샘플의 학습을 이어서 수행한 것인가요? 손실 감소로 어디까지 확인할 수 있나요?

#### 제출 결과

- 순전파·기울기·새 파라미터·손실 비교 출력
- 별도 기울기 전달 복습 결과
- 손계산 및 Q1~Q5 답변


In [ ]:
# 제공 코드: 원래 초기값으로 다시 시작합니다.
task_x = df["cement"].to_numpy()[2] / 100.0
task_y = df["concrete_compressive_strength"].to_numpy()[2]
task_w1 = 0.5
task_b1 = 0.1
task_w2 = 20.0
task_b2 = 20.0
task_lr = 0.0001

# 아래부터 작성하세요.
# 순전파 → 손실 → 역전파 → 네 파라미터 수정 → 새 손실 확인
# 별도 복습: 1.0 * (0.1 ** 3), 1.0 * (2.0 ** 3)


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. 수정 전 z1, h1, 출력, 오차, 손실은 각각 얼마인가요?**

- 답변: 

**Q2. 네 파라미터의 기울기는 각각 얼마인가요? 실습의 첫 샘플과 부호가 다른 이유는 무엇인가요?**

- 답변: 

**Q3. 네 파라미터를 한 번 수정한 값과 수정 후 출력·손실은 얼마인가요? 파라미터가 증가했나요, 감소했나요?**

- 답변: 

**Q4. 기울기 1에 계수 0.1과 2를 세 번 곱한 결과는 각각 얼마이며, 어떤 현상을 설명하나요?**

- 답변: 

**Q5. 이번 과제가 첫 샘플의 학습을 이어서 수행한 것인가요? 손실 감소로 어디까지 확인할 수 있나요?**

- 답변:


---

## 실습 마무리

**1. 순전파와 역전파의 진행 방향과 역할은 어떻게 다른가요?**

- 답변: 

**2. 역전파와 파라미터 수정은 같은 작업인가요?**

- 답변: 

**3. 기울기가 음수일 때 기존 값에서 lr×기울기를 빼면 어떻게 되나요?**

- 답변: 

**4. 기울기 소실과 폭주를 각각 설명하세요.**

- 답변: 

**5. 같은 출력 기울기 1에서 0.1과 2를 반복해서 곱하는 이유는 무엇인가요?**

- 답변: 

**6. 이번 실습으로 전체 콘크리트 데이터의 모델 성능을 확인했나요?**

- 답변:
